In [17]:
import sys
!{sys.executable} -m pip uninstall -y vllm vllm-flash-attn



Found existing installation: vllm 0.5.4
Uninstalling vllm-0.5.4:
  Successfully uninstalled vllm-0.5.4
Found existing installation: vllm-flash-attn 2.6.1
Uninstalling vllm-flash-attn-2.6.1:
  Successfully uninstalled vllm-flash-attn-2.6.1


In [9]:
import subprocess
import os

env = os.environ.copy()
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"

print("Launching Llama-2-7b-hf Pipeline (Prune -> Eval) on GPU 1...")
env["CUDA_VISIBLE_DEVICES"] = "1"
llama_cmd = (
    "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py "
    "--model meta-llama/Llama-2-7b-hf --structured-ratio 0.25 --obs-reconstruct "
    "--save-dir pruned_models/llama2_7b "
    "&& /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_eval.py "
    "--model meta-llama/Llama-2-7b-hf --model-path pruned_models/llama2_7b --sobp-tasks"
)
llama_process = subprocess.Popen(llama_cmd, shell=True, env=env, cwd=work_dir, stdout=open(f"{work_dir}/pipeline_llama2_7b.log", "w"), stderr=subprocess.STDOUT)

print("Launching Phi-2 Pipeline (Prune -> Eval) on GPU 2...")
env["CUDA_VISIBLE_DEVICES"] = "2"
phi2_cmd = (
    "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py "
    "--model microsoft/phi-2 --structured-ratio 0.25 --obs-reconstruct "
    "--save-dir pruned_models/phi2 "
    "&& /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_eval.py "
    "--model microsoft/phi-2 --model-path pruned_models/phi2 --sobp-tasks"
)
phi2_process = subprocess.Popen(phi2_cmd, shell=True, env=env, cwd=work_dir, stdout=open(f"{work_dir}/pipeline_phi2.log", "w"), stderr=subprocess.STDOUT)

print("Launching OPT-6.7b Pipeline (Prune -> Eval) on GPU 3...")
env["CUDA_VISIBLE_DEVICES"] = "3"
opt_cmd = (
    "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py "
    "--model facebook/opt-6.7b --structured-ratio 0.25 --obs-reconstruct "
    "--save-dir pruned_models/opt_6.7b "
    "&& /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_eval.py "
    "--model facebook/opt-6.7b --model-path pruned_models/opt_6.7b --sobp-tasks"
)
opt_process = subprocess.Popen(opt_cmd, shell=True, env=env, cwd=work_dir, stdout=open(f"{work_dir}/pipeline_opt_6.7b.log", "w"), stderr=subprocess.STDOUT)

print("\nAll 3 End-to-End pipelines launched!")
print(f"PIDs -> LLaMA: {llama_process.pid}, Phi-2: {phi2_process.pid}, OPT: {opt_process.pid}")
print("You can view the full combined progress live by running:")
print(f"!tail -f {work_dir}/pipeline_llama2_7b.log")


Launching Llama-2-7b-hf Pipeline (Prune -> Eval) on GPU 1...
Launching Phi-2 Pipeline (Prune -> Eval) on GPU 2...
Launching OPT-6.7b Pipeline (Prune -> Eval) on GPU 3...

All 3 End-to-End pipelines launched!
PIDs -> LLaMA: 4130970, Phi-2: 4130972, OPT: 4130974
You can view the full combined progress live by running:
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/pipeline_llama2_7b.log


In [35]:
import subprocess
import os

env = os.environ.copy()
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"

print("1. Pre-caching missing datasets sequentially so HuggingFace doesn't ban us...")
cache_script = """
from datasets import load_dataset
for d, sub in [('piqa', None), ('hellaswag', None), ('ai2_arc', 'ARC-Challenge'), 
               ('winogrande', 'winogrande_xl'), ('super_glue', 'boolq'), ('openbookqa', 'main')]:
    try:
        load_dataset(d, sub)
    except Exception as e:
        print(f"Skipped {d}: {e}")
"""
subprocess.run(["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "-c", cache_script], env=env)
print("Cache ready!\n")

print("2. Launching all 3 Evaluations in Parallel...")
print("Evaluating LLaMA-2...")
env["CUDA_VISIBLE_DEVICES"] = "1"
subprocess.Popen(
    ["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", 
     "--model", "meta-llama/Llama-2-7b-hf", "--model-path", "pruned_models/llama2_7b", "--sobp-tasks"],
    env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_llama2_7b.log", "w"), stderr=subprocess.STDOUT
)

print("Evaluating Phi-2...")
env["CUDA_VISIBLE_DEVICES"] = "2"
subprocess.Popen(
    ["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", 
     "--model", "microsoft/phi-2", "--model-path", "pruned_models/phi2", "--sobp-tasks"],
    env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_phi2.log", "w"), stderr=subprocess.STDOUT
)

print("Evaluating OPT-6.7b...")
env["CUDA_VISIBLE_DEVICES"] = "3"
subprocess.Popen(
    ["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", 
     "--model", "facebook/opt-6.7b", "--model-path", "pruned_models/opt_6.7b", "--sobp-tasks"],
    env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_opt_6.7b.log", "w"), stderr=subprocess.STDOUT
)

print("Pipelines successfully launched! You can monitor them via: !tail -f eval_llama2_7b.log")


1. Pre-caching missing datasets sequentially so HuggingFace doesn't ban us...


Generating test split: 100%|██████████| 500/500 [00:00<00:00, 165795.87 examples/s]


Cache ready!

2. Launching all 3 Evaluations in Parallel...
Evaluating LLaMA-2...
Evaluating Phi-2...
Evaluating OPT-6.7b...
Pipelines successfully launched! You can monitor them via: !tail -f eval_llama2_7b.log


In [37]:
!tail -1000 /nlsasfs/home/isea/isea28/SARATHI-E9FD/eval_llama2_7b.log

2026-08-26 19:58:49  INFO      [SARATHI] Using SoBP 7-task evaluation set.
2026-08-26 19:58:49  INFO      [SARATHI] Loading model from: /nlsasfs/home/isea/isea28/SARATHI-E9FD/pruned_models/llama2_7b
2026-08-26 19:58:49  INFO      Note: detected 256 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
2026-08-26 19:58:49  INFO      Note: NumExpr detected 256 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
2026-08-26 19:58:49  INFO      NumExpr defaulting to 16 threads.
2026-08-26 19:59:11  WARNING   Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.
2026-08-26 19:59:16  WARNING   Some tasks could not be loaded due to missing dependencies. Run with `--verbosity DEBUG` for full details.
Loading checkpoint shards: 100%|██████████| 3/3 [01:18<00:00, 26.21s/it]
2026-08-26 20:00:51  WARNING   `pretrained` model kwarg is not of type `str`. Many other model arguments may 

In [38]:
import subprocess
import os

env = os.environ.copy()
print("Downloading missing datasets (Rate limit is now lifted!)...")
cache_script = """
from datasets import load_dataset
for d, sub in [('super_glue', 'boolq'), ('openbookqa', 'main')]:
    try:
        load_dataset(d, sub)
        print(f"Successfully cached {d}!")
    except Exception as e:
        print(f"Failed to cache {d}: {e}")
"""
subprocess.run(["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "-c", cache_script], env=env)


Successfully cached super_glue!


Successfully cached openbookqa!


CompletedProcess(args=['/nlsasfs/home/isea/isea28/venv_jupyter/bin/python', '-c', '\nfrom datasets import load_dataset\nfor d, sub in [(\'super_glue\', \'boolq\'), (\'openbookqa\', \'main\')]:\n    try:\n        load_dataset(d, sub)\n        print(f"Successfully cached {d}!")\n    except Exception as e:\n        print(f"Failed to cache {d}: {e}")\n'], returncode=0)

In [40]:
import subprocess
import os
import time

env = os.environ.copy()
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"

print("Evaluating LLaMA-2 on GPU 1...")
env["CUDA_VISIBLE_DEVICES"] = "1"
subprocess.Popen(["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", "--model", "meta-llama/Llama-2-7b-hf", "--model-path", "pruned_models/llama2_7b", "--sobp-tasks"], env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_llama2_7b.log", "w"), stderr=subprocess.STDOUT)

print("Sleeping for 60 seconds to avoid HuggingFace API bans...")
time.sleep(60)

print("Evaluating Phi-2 on GPU 2...")
env["CUDA_VISIBLE_DEVICES"] = "2"
subprocess.Popen(["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", "--model", "microsoft/phi-2", "--model-path", "pruned_models/phi2", "--sobp-tasks"], env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_phi2.log", "w"), stderr=subprocess.STDOUT)

print("Sleeping for 60 seconds to avoid HuggingFace API bans...")
time.sleep(60)

print("Evaluating OPT-6.7b on GPU 3...")
env["CUDA_VISIBLE_DEVICES"] = "3"
subprocess.Popen(["/nlsasfs/home/isea/isea28/venv_jupyter/bin/python", "sarathi_eval.py", "--model", "facebook/opt-6.7b", "--model-path", "pruned_models/opt_6.7b", "--sobp-tasks"], env=env, cwd=work_dir, stdout=open(f"{work_dir}/eval_opt_6.7b.log", "w"), stderr=subprocess.STDOUT)

print("\nAll 3 launched securely in the background!")


Evaluating LLaMA-2 on GPU 1...
Sleeping for 60 seconds to avoid HuggingFace API bans...
Evaluating Phi-2 on GPU 2...
Sleeping for 60 seconds to avoid HuggingFace API bans...
Evaluating OPT-6.7b on GPU 3...

All 3 launched securely in the background!


In [41]:
import subprocess
import os
import time

env = os.environ.copy()
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"

models = {
    "mistral": ("mistralai/Mistral-7B-v0.1", "1"),
    "llama3": ("meta-llama/Meta-Llama-3-8B", "2"),
    "opt": ("facebook/opt-6.7b", "3")
}

ratios = ["0.20", "0.40"]
variants = {"E": "NMF", "B": "Wanda"}

for alias, (path, gpu) in models.items():
    script_content = f"#!/bin/bash\nexport CUDA_VISIBLE_DEVICES={gpu}\ncd {work_dir}\n\n"
    
    for r in ratios:
        for v_code, v_name in variants.items():
            run_name = f"{alias}_{v_name}_{r}"
            out_dir = f"pruned_models/{run_name}"
            
            script_content += f"echo '========================================'\n"
            script_content += f"echo 'Starting Pruning: {run_name}'\n"
            script_content += f"{python_exec} sarathi_main.py --model {path} --structured-ratio {r} --variant {v_code} --obs-reconstruct --save-dir {out_dir} > {run_name}_prune.log 2>&1\n"
            
            script_content += f"echo 'Starting Eval: {run_name}'\n"
            script_content += f"{python_exec} sarathi_eval.py --model {path} --model-path {out_dir} --sobp-tasks --ppl > {run_name}_eval.log 2>&1\n\n"
            
    script_file = f"run_{alias}_sweep.sh"
    with open(f"{work_dir}/{script_file}", "w") as f:
        f.write(script_content)
    
    print(f"Launching {alias} sweep on GPU {gpu}...")
    subprocess.Popen(["bash", script_file], cwd=work_dir)
    
    # Stagger launches to prevent Hugging Face API rate limits
    time.sleep(60) 

print("All 3 grid sweeps successfully launched in the background!")
print("You can monitor them with:")
print("!tail -f mistral_NMF_0.20_prune.log")


Launching mistral sweep on GPU 1...
Starting Pruning: mistral_NMF_0.20
Launching llama3 sweep on GPU 2...
Starting Pruning: llama3_NMF_0.20
Launching opt sweep on GPU 3...
Starting Pruning: opt_NMF_0.20
All 3 grid sweeps successfully launched in the background!
You can monitor them with:
!tail -f mistral_NMF_0.20_prune.log
Starting Eval: mistral_NMF_0.20
Starting Eval: llama3_NMF_0.20
Starting Eval: opt_NMF_0.20
Starting Pruning: mistral_Wanda_0.20
Starting Pruning: llama3_Wanda_0.20
Starting Pruning: opt_Wanda_0.20
Starting Eval: mistral_Wanda_0.20
Starting Eval: llama3_Wanda_0.20
Starting Eval: opt_Wanda_0.20
Starting Pruning: mistral_NMF_0.40
Starting Pruning: llama3_NMF_0.40
Starting Pruning: opt_NMF_0.40
Starting Eval: mistral_NMF_0.40
Starting Eval: llama3_NMF_0.40
Starting Pruning: mistral_Wanda_0.40
Starting Eval: opt_NMF_0.40
Starting Pruning: llama3_Wanda_0.40
Starting Pruning: opt_Wanda_0.40
Starting Eval: mistral_Wanda_0.40
Starting Eval: llama3_Wanda_0.40
Starting Eval: op

In [42]:
import subprocess
import os
import time
from datetime import datetime

env = os.environ.copy()
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"

# Mapped to GPUs 0, 1, and 2
models = {
    "mistral": ("mistralai/Mistral-7B-v0.1", "0"),
    "llama3": ("meta-llama/Meta-Llama-3-8B", "1"),
    "opt": ("facebook/opt-6.7b", "2")
}

ratios = ["0.20", "0.40"]
variants = {"E": "NMF", "B": "Wanda"}

# Ensure the logs directory exists
os.makedirs(f"{work_dir}/logs", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for alias, (path, gpu) in models.items():
    script_content = f"#!/bin/bash\nexport CUDA_VISIBLE_DEVICES={gpu}\ncd {work_dir}\n\n"
    
    for r in ratios:
        for v_code, v_name in variants.items():
            run_name = f"{alias}_{v_name}_{r}"
            out_dir = f"pruned_models/{run_name}"
            
            script_content += f"echo '========================================'\n"
            script_content += f"echo 'Starting Pruning: {run_name}'\n"
            # ADDED --adaptive FLAG AND ROUTED LOGS TO logs/ DIRECTORY
            script_content += f"{python_exec} sarathi_main.py --model {path} --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
            
            script_content += f"echo 'Starting Eval: {run_name}'\n"
            script_content += f"{python_exec} sarathi_eval.py --model {path} --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n\n"
            
    script_file = f"run_{alias}_sweep.sh"
    with open(f"{work_dir}/{script_file}", "w") as f:
        f.write(script_content)
    
    print(f"Launching {alias} sweep on GPU {gpu}...")
    subprocess.Popen(["bash", script_file], cwd=work_dir)
    
    # Stagger launches to prevent Hugging Face API rate limits
    time.sleep(60) 

print("All 3 grid sweeps successfully launched in the background!")
print("You can monitor them with:")
print(f"!tail -f logs/{timestamp}_mistral_NMF_0.20_prune.log")


Launching mistral sweep on GPU 0...
Starting Pruning: mistral_NMF_0.20
Launching llama3 sweep on GPU 1...
Starting Pruning: llama3_NMF_0.20
Launching opt sweep on GPU 2...
Starting Pruning: opt_NMF_0.20
All 3 grid sweeps successfully launched in the background!
You can monitor them with:
!tail -f logs/20260827_115406_mistral_NMF_0.20_prune.log
Starting Eval: llama3_NMF_0.20
Starting Eval: mistral_NMF_0.20
Starting Eval: opt_NMF_0.20
Starting Pruning: llama3_Wanda_0.20
Starting Pruning: mistral_Wanda_0.20
Starting Pruning: opt_Wanda_0.20
Starting Eval: llama3_Wanda_0.20
Starting Eval: mistral_Wanda_0.20
Starting Eval: opt_Wanda_0.20
Starting Pruning: llama3_NMF_0.40
Starting Pruning: mistral_NMF_0.40
Starting Pruning: opt_NMF_0.40
Starting Eval: llama3_NMF_0.40
Starting Eval: mistral_NMF_0.40
Starting Eval: opt_NMF_0.40
Starting Pruning: llama3_Wanda_0.40
Starting Pruning: mistral_Wanda_0.40
Starting Pruning: opt_Wanda_0.40
Starting Eval: llama3_Wanda_0.40
Starting Eval: mistral_Wanda_0

In [43]:
import subprocess
import os
from datetime import datetime

work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Model configuration: (huggingface_path, gpu_id, extra_args)
models = {
    "mistral": ("mistralai/Mistral-7B-v0.1", "0", ""),
    "llama3": ("meta-llama/Meta-Llama-3-8B", "1", ""),
    "opt": ("facebook/opt-6.7b", "2", "--calib-dataset c4") # C4 specifically for OPT
}

ratios = ["0.20", "0.40"]
variants = {"E": "NMF", "B": "Wanda"}

# Generate and launch a bash script for each model
for alias, (path, gpu, extra_args) in models.items():
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=1
cd {work_dir}

"""
    for r in ratios:
        for v_code, v_name in variants.items():
            run_name = f"{alias}_{v_name}_{r}_final"
            out_dir = f"pruned_models/{run_name}"
            
            # Prune
            script_content += f"echo '========================================'\n"
            script_content += f"echo 'Starting FINAL Pruning: {run_name}'\n"
            script_content += f"{python_exec} sarathi_main.py --model {path} --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct {extra_args} --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
            
            # Evaluate
            script_content += f"echo 'Finished Pruning. Starting FINAL Evaluation: {run_name}'\n"
            script_content += f"{python_exec} sarathi_eval.py --model {path} --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n"
            script_content += f"echo 'Finished Evaluation: {run_name}'\n\n"

    script_file = f"run_final_{alias}.sh"
    with open(f"{work_dir}/{script_file}", "w") as f:
        f.write(script_content)
    
    # Make executable and run in background
    os.chmod(f"{work_dir}/{script_file}", 0o755)
    print(f"Launching {alias} on GPU {gpu}...")
    subprocess.Popen(["bash", script_file], cwd=work_dir)

print("\n🚀 ALL FINAL SWEEPS SUCCESSFULLY LAUNCHED IN JUPYTER!")
print(f"You can monitor progress in Jupyter with:")
print(f"!tail -f logs/{timestamp}_mistral_NMF_0.20_final_prune.log")


Launching mistral on GPU 0...
Launching llama3 on GPU 1...
Starting FINAL Pruning: mistral_NMF_0.20_final
Launching opt on GPU 2...
Starting FINAL Pruning: llama3_NMF_0.20_final

🚀 ALL FINAL SWEEPS SUCCESSFULLY LAUNCHED IN JUPYTER!
You can monitor progress in Jupyter with:
!tail -f logs/20260827_191144_mistral_NMF_0.20_final_prune.log
Starting FINAL Pruning: opt_NMF_0.20_final
Finished Pruning. Starting FINAL Evaluation: opt_NMF_0.20_final
Finished Evaluation: opt_NMF_0.20_final
Starting FINAL Pruning: opt_Wanda_0.20_final
Finished Pruning. Starting FINAL Evaluation: opt_Wanda_0.20_final
Finished Evaluation: opt_Wanda_0.20_final
Starting FINAL Pruning: opt_NMF_0.40_final
Finished Pruning. Starting FINAL Evaluation: opt_NMF_0.40_final
Finished Evaluation: opt_NMF_0.40_final
Starting FINAL Pruning: opt_Wanda_0.40_final
Finished Pruning. Starting FINAL Evaluation: opt_Wanda_0.40_final
Finished Evaluation: opt_Wanda_0.40_final


In [44]:
import subprocess
import os
from datetime import datetime

work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = "facebook/opt-6.7b"
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

# Split OPT across GPU 2 and GPU 3 by ratio
opt_splits = {
    "gpu2": ("2", "0.20"),  # GPU 2 → 20% sparsity (NMF + Wanda)
    "gpu3": ("3", "0.40"),  # GPU 3 → 40% sparsity (NMF + Wanda)
}

variants = {"E": "NMF", "B": "Wanda"}

for split_name, (gpu, r) in opt_splits.items():
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=1
export SARATHI_C4_PATH={c4_path}
cd {work_dir}

"""
    for v_code, v_name in variants.items():
        run_name = f"opt_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        
        script_content += f"echo '========================================'\n"
        script_content += f"echo 'Starting: {run_name} on GPU {gpu}'\n"
        script_content += f"{python_exec} sarathi_main.py --model {model_path} --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --calib-dataset c4 --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        script_content += f"echo 'Evaluating: {run_name}'\n"
        script_content += f"{python_exec} sarathi_eval.py --model {model_path} --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n"
        script_content += f"echo 'Done: {run_name}'\n\n"

    script_file = f"run_opt_{split_name}.sh"
    with open(f"{work_dir}/{script_file}", "w") as f:
        f.write(script_content)
    
    os.chmod(f"{work_dir}/{script_file}", 0o755)
    print(f"Launching OPT {r} sparsity on GPU {gpu}...")
    subprocess.Popen(["bash", script_file], cwd=work_dir)

print(f"\n🚀 OPT re-runs launched on GPU 2 (20%) and GPU 3 (40%)!")
print(f"Monitor with:")
print(f"!tail -f logs/{timestamp}_opt_NMF_0.20_final_prune.log")
print(f"!tail -f logs/{timestamp}_opt_NMF_0.40_final_prune.log")


Launching OPT 0.20 sparsity on GPU 2...
Launching OPT 0.40 sparsity on GPU 3...
Starting: opt_NMF_0.20_final on GPU 2

🚀 OPT re-runs launched on GPU 2 (20%) and GPU 3 (40%)!
Monitor with:
!tail -f logs/20260827_193332_opt_NMF_0.20_final_prune.log
!tail -f logs/20260827_193332_opt_NMF_0.40_final_prune.log
Starting: opt_NMF_0.40_final on GPU 3
Finished Pruning. Starting FINAL Evaluation: llama3_NMF_0.20_final
Finished Pruning. Starting FINAL Evaluation: mistral_NMF_0.20_final
Finished Evaluation: mistral_NMF_0.20_final
Starting FINAL Pruning: mistral_Wanda_0.20_final
Finished Evaluation: llama3_NMF_0.20_final
Starting FINAL Pruning: llama3_Wanda_0.20_final
Evaluating: opt_NMF_0.20_final
Evaluating: opt_NMF_0.40_final
Done: opt_NMF_0.20_final
Starting: opt_Wanda_0.20_final on GPU 2
Done: opt_NMF_0.40_final
Starting: opt_Wanda_0.40_final on GPU 3
Finished Pruning. Starting FINAL Evaluation: mistral_Wanda_0.20_final
Finished Pruning. Starting FINAL Evaluation: llama3_Wanda_0.20_final
Finish

In [47]:
import subprocess
import os
import time
from datetime import datetime

# ================= Configuration =================
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

# Models mapped to GPUs (Mistral:0, LLaMA-3:1, OPT(20%):2, OPT(40%):3)
variants = {"E": "NMF", "B": "Wanda"}
ratios = ["0.20", "0.40"]

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

scripts_to_launch = []

# ================= 1. Mistral (GPU 0) =================
mistral_script = f"#!/bin/bash\nexport CUDA_VISIBLE_DEVICES=0\nexport HF_HUB_OFFLINE=0\ncd {work_dir}\n\n"
for r in ratios:
    for v_code, v_name in variants.items():
        run_name = f"mistral_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        mistral_script += f"echo 'Starting: {run_name}'\n"
        # Uses default wikitext for calibration
        mistral_script += f"{python_exec} sarathi_main.py --model mistralai/Mistral-7B-v0.1 --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        mistral_script += f"{python_exec} sarathi_eval.py --model mistralai/Mistral-7B-v0.1 --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n\n"

script_path = f"{work_dir}/run_mistral_sweep.sh"
with open(script_path, "w") as f: f.write(mistral_script)
os.chmod(script_path, 0o755)
scripts_to_launch.append(script_path)

# ================= 2. LLaMA-3 (GPU 1) =================
llama3_script = f"#!/bin/bash\nexport CUDA_VISIBLE_DEVICES=1\nexport HF_HUB_OFFLINE=0\ncd {work_dir}\n\n"
for r in ratios:
    for v_code, v_name in variants.items():
        run_name = f"llama3_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        llama3_script += f"echo 'Starting: {run_name}'\n"
        # Uses default wikitext for calibration
        llama3_script += f"{python_exec} sarathi_main.py --model meta-llama/Meta-Llama-3-8B --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        llama3_script += f"{python_exec} sarathi_eval.py --model meta-llama/Meta-Llama-3-8B --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n\n"

script_path = f"{work_dir}/run_llama3_sweep.sh"
with open(script_path, "w") as f: f.write(llama3_script)
os.chmod(script_path, 0o755)
scripts_to_launch.append(script_path)

# ================= 3. OPT-6.7B (GPU 2 for 20%, GPU 3 for 40%) =================
opt_splits = {"gpu2": ("2", "0.20"), "gpu3": ("3", "0.40")}
for split_name, (gpu, r) in opt_splits.items():
    opt_script = f"#!/bin/bash\nexport CUDA_VISIBLE_DEVICES={gpu}\nexport HF_HUB_OFFLINE=0\nexport SARATHI_C4_PATH={c4_path}\ncd {work_dir}\n\n"
    for v_code, v_name in variants.items():
        run_name = f"opt_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        opt_script += f"echo 'Starting: {run_name}'\n"
        # Explicitly using --calib-dataset c4 for OPT
        opt_script += f"{python_exec} sarathi_main.py --model facebook/opt-6.7b --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --calib-dataset c4 --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        opt_script += f"{python_exec} sarathi_eval.py --model facebook/opt-6.7b --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n\n"
    
    script_path = f"{work_dir}/run_opt_{split_name}.sh"
    with open(script_path, "w") as f: f.write(opt_script)
    os.chmod(script_path, 0o755)
    scripts_to_launch.append(script_path)

# ================= Execution =================
print("Launching sweeps across GPUs in the background...")
for script in scripts_to_launch:
    print(f"Executing: {os.path.basename(script)}")
    subprocess.Popen(["bash", script], cwd=work_dir)
    time.sleep(1) # Small gap between launches

print("\nAll pipelines launched successfully!")
print("Aap apne Jupyter Notebook ke doosre cell mein ye command daal kar logs monitor kar sakte ho:")
print(f"!tail -f {work_dir}/logs/{timestamp}_opt_NMF_0.20_final_prune.log")


Launching sweeps across GPUs in the background...
Executing: run_mistral_sweep.sh
Starting: mistral_NMF_0.20_final
Executing: run_llama3_sweep.sh
Starting: llama3_NMF_0.20_final
Executing: run_opt_gpu2.sh
Starting: opt_NMF_0.20_final
Executing: run_opt_gpu3.sh
Starting: opt_NMF_0.40_final

All pipelines launched successfully!
Aap apne Jupyter Notebook ke doosre cell mein ye command daal kar logs monitor kar sakte ho:
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260827_215553_opt_NMF_0.20_final_prune.log
Starting: mistral_Wanda_0.20_final
Starting: llama3_Wanda_0.20_final
Starting: mistral_NMF_0.40_final
Starting: llama3_NMF_0.40_final
Starting: llama3_Wanda_0.20_final


In [1]:
import subprocess
import os
import time
from datetime import datetime

# ================= Configuration =================
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

# 4 Tasks mapped precisely to 4 GPUs for maximum parallelization
tasks = [
    {"gpu": 0, "ratio": "0.20", "variant_code": "E", "variant_name": "NMF"},
    {"gpu": 1, "ratio": "0.20", "variant_code": "B", "variant_name": "Wanda"},
    {"gpu": 2, "ratio": "0.40", "variant_code": "E", "variant_name": "NMF"},
    {"gpu": 3, "ratio": "0.40", "variant_code": "B", "variant_name": "Wanda"}
]

scripts_to_launch = []

for task in tasks:
    gpu = task["gpu"]
    r = task["ratio"]
    v_code = task["variant_code"]
    v_name = task["variant_name"]
    
    run_name = f"opt_{v_name}_rank16_{r}_final"
    out_dir = f"pruned_models/{run_name}"
    
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=0
export SARATHI_C4_PATH={c4_path}
cd {work_dir}

echo 'Starting: {run_name} on GPU {gpu}'
{python_exec} sarathi_main.py --model facebook/opt-6.7b --structured-ratio {r} --variant {v_code} --nmf-rank 16 --adaptive --obs-reconstruct --calib-dataset c4 --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1

echo 'Starting Evaluation: {run_name}'
{python_exec} sarathi_eval.py --model facebook/opt-6.7b --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1
echo 'Finished: {run_name}'
"""
    
    script_path = f"{work_dir}/run_opt_gpu{gpu}_{timestamp}.sh"
    with open(script_path, "w") as f: 
        f.write(script_content)
    os.chmod(script_path, 0o755)
    scripts_to_launch.append(script_path)

# ================= Execution =================
print("Launching OPT sweeps across 4 GPUs...")
for script in scripts_to_launch:
    print(f"Executing: {os.path.basename(script)}")
    subprocess.Popen(["bash", script], cwd=work_dir)
    time.sleep(1) # Small gap between launches

print("\nAll 4 OPT pipelines launched successfully in parallel!")
print("Aap kisi ek specific GPU ka log dekhne ke liye ye run kar sakte ho:")
print(f"!tail -f {work_dir}/logs/{timestamp}_opt_NMF_rank16_0.20_final_prune.log")


Launching OPT sweeps across 4 GPUs...
Executing: run_opt_gpu0_20260828_101727.sh
Starting: opt_NMF_rank16_0.20_final on GPU 0
Executing: run_opt_gpu1_20260828_101727.sh
Starting: opt_Wanda_rank16_0.20_final on GPU 1
Executing: run_opt_gpu2_20260828_101727.sh
Starting: opt_NMF_rank16_0.40_final on GPU 2
Executing: run_opt_gpu3_20260828_101727.sh
Starting: opt_Wanda_rank16_0.40_final on GPU 3

All 4 OPT pipelines launched successfully in parallel!
Aap kisi ek specific GPU ka log dekhne ke liye ye run kar sakte ho:
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260828_101727_opt_NMF_rank16_0.20_final_prune.log


In [2]:
import subprocess
import os
import time
from datetime import datetime

# ================= Configuration =================
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

# ONLY running the Wanda tasks (GPUs 1 and 3)
tasks = [
    {"gpu": 1, "ratio": "0.20", "variant_code": "B", "variant_name": "Wanda"},
    {"gpu": 3, "ratio": "0.40", "variant_code": "B", "variant_name": "Wanda"}
]

scripts_to_launch = []

for task in tasks:
    gpu = task["gpu"]
    r = task["ratio"]
    v_code = task["variant_code"]
    v_name = task["variant_name"]
    
    run_name = f"opt_{v_name}_rank16_{r}_final"
    out_dir = f"pruned_models/{run_name}"
    
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=0
export SARATHI_C4_PATH={c4_path}
cd {work_dir}

echo 'Starting: {run_name} on GPU {gpu}'
# CHANGED: --calib-dataset wikitext
{python_exec} sarathi_main.py --model facebook/opt-6.7b --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct --calib-dataset wikitext --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1

echo 'Starting Evaluation: {run_name}'
{python_exec} sarathi_eval.py --model facebook/opt-6.7b --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1
echo 'Finished: {run_name}'
"""
    
    script_path = f"{work_dir}/run_opt_gpu{gpu}_{timestamp}.sh"
    with open(script_path, "w") as f: 
        f.write(script_content)
    os.chmod(script_path, 0o755)
    scripts_to_launch.append(script_path)

# ================= Execution =================
print("Launching ONLY OPT Wanda sweeps across GPUs 1 and 3...")
for script in scripts_to_launch:
    print(f"Executing: {os.path.basename(script)}")
    subprocess.Popen(["bash", script], cwd=work_dir)
    time.sleep(1) # Small gap between launches

print("\nWanda pipelines launched successfully on GPUs 1 and 3!")
print("Monitor GPU 1 (20%):")
print(f"!tail -f {work_dir}/logs/{timestamp}_opt_Wanda_rank16_0.20_final_prune.log")
print("Monitor GPU 3 (40%):")
print(f"!tail -f {work_dir}/logs/{timestamp}_opt_Wanda_rank16_0.40_final_prune.log")


Launching ONLY OPT Wanda sweeps across GPUs 1 and 3...
Executing: run_opt_gpu1_20260828_112221.sh
Starting: opt_Wanda_rank16_0.20_final on GPU 1
Executing: run_opt_gpu3_20260828_112221.sh
Starting: opt_Wanda_rank16_0.40_final on GPU 3

Wanda pipelines launched successfully on GPUs 1 and 3!
Monitor GPU 1 (20%):
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260828_112221_opt_Wanda_rank16_0.20_final_prune.log
Monitor GPU 3 (40%):
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260828_112221_opt_Wanda_rank16_0.40_final_prune.log
Starting Evaluation: opt_Wanda_rank16_0.20_final


/nlsasfs/home/isea/isea28/SARATHI-E9FD/run_opt_gpu2_20260828_101727.sh: line 8: 2799847 Killed                  /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py --model facebook/opt-6.7b --structured-ratio 0.40 --variant E --nmf-rank 16 --adaptive --obs-reconstruct --calib-dataset c4 --save-dir pruned_models/opt_NMF_rank16_0.40_final > logs/20260828_101727_opt_NMF_rank16_0.40_final_prune.log 2>&1


Starting Evaluation: opt_NMF_rank16_0.40_final


/nlsasfs/home/isea/isea28/SARATHI-E9FD/run_opt_gpu0_20260828_101727.sh: line 8: 2799680 Killed                  /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py --model facebook/opt-6.7b --structured-ratio 0.20 --variant E --nmf-rank 16 --adaptive --obs-reconstruct --calib-dataset c4 --save-dir pruned_models/opt_NMF_rank16_0.20_final > logs/20260828_101727_opt_NMF_rank16_0.20_final_prune.log 2>&1


Starting Evaluation: opt_NMF_rank16_0.20_final


/nlsasfs/home/isea/isea28/SARATHI-E9FD/run_opt_gpu3_20260828_112221.sh: line 9: 2871191 Killed                  /nlsasfs/home/isea/isea28/venv_jupyter/bin/python sarathi_main.py --model facebook/opt-6.7b --structured-ratio 0.40 --variant B --adaptive --obs-reconstruct --calib-dataset wikitext --save-dir pruned_models/opt_Wanda_rank16_0.40_final > logs/20260828_112221_opt_Wanda_rank16_0.40_final_prune.log 2>&1


Starting Evaluation: opt_Wanda_rank16_0.40_final
Finished: opt_NMF_rank16_0.40_final
Finished: opt_NMF_rank16_0.20_final


In [7]:
import subprocess
import os
import time
from datetime import datetime

# ================= CRITICAL FIX: Auto-Kill Previous Runs =================
print("Cleaning up old processes and freeing GPU memory...")
os.system("pkill -u isea28 -f 'sarathi_main'")
os.system("pkill -u isea28 -f 'sarathi_eval'")
time.sleep(3) # Wait for GPUs to clear
print("Cleanup complete!")

# ================= Configuration =================
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
# WAPAS VENV JUPYTER (dhruv_env broken hai isliye yahi use karna hai 100%!)
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

# Organize tasks by GPU. Each list runs SEQUENTIALLY on its GPU to prevent OOM
tasks_by_gpu = {
    # GPU 0: Mistral (needs wikitext & obs-damping 0.01)
    0: [
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.20", "E", "NMF", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.40", "E", "NMF", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.20", "B", "Wanda", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.40", "B", "Wanda", "--calib-dataset wikitext --obs-damping 0.01")
    ],
    # GPU 1: LLaMA-3 (needs wikitext)
    1: [
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "E", "NMF", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "E", "NMF", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "B", "Wanda", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "B", "Wanda", "--calib-dataset wikitext")
    ],
    # GPU 2: OPT NMF (needs c4)
    2: [
        ("facebook/opt-6.7b", "opt", "0.20", "E", "NMF", "--calib-dataset c4"),
        ("facebook/opt-6.7b", "opt", "0.40", "E", "NMF", "--calib-dataset c4")
    ],
    # GPU 3: OPT WANDA (needs wikitext)
    3: [
        ("facebook/opt-6.7b", "opt", "0.20", "B", "Wanda", "--calib-dataset wikitext"),
        ("facebook/opt-6.7b", "opt", "0.40", "B", "Wanda", "--calib-dataset wikitext")
    ]
}

scripts_to_launch = []

for gpu, tasks in tasks_by_gpu.items():
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=0
export SARATHI_C4_PATH={c4_path}
cd {work_dir}

"""
    for model, alias, r, v_code, v_name, extra_args in tasks:
        # Match original run name formatting
        run_name = f"{alias}_{v_name}_rank16_{r}_final" if alias == "opt" else f"{alias}_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        
        script_content += f"echo 'Starting Pruning: {run_name} on GPU {gpu}'\n"
        script_content += f"{python_exec} sarathi_main.py --model {model} --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct {extra_args} --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        
        script_content += f"echo 'Starting Evaluation: {run_name}'\n"
        script_content += f"{python_exec} sarathi_eval.py --model {model} --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n"
        script_content += f"echo 'Finished: {run_name}'\n\n"

    script_path = f"{work_dir}/run_gpu{gpu}_{timestamp}.sh"
    with open(script_path, "w") as f: 
        f.write(script_content)
    os.chmod(script_path, 0o755)
    scripts_to_launch.append((gpu, script_path))

# ================= Execution =================
print("Launching sweeps sequentially per GPU...")
for gpu, script in scripts_to_launch:
    print(f"Executing on GPU {gpu}: {os.path.basename(script)}")
    # Use nohup so jupyter won't kill it if disconnected
    subprocess.Popen(["nohup", "bash", script], cwd=work_dir, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)

print("\nALL PIPELINES LAUNCHED SAFELY WITH NOHUP!")
print("To check progress on GPU 0 (Mistral NMF 20%):")
print(f"!tail -f {work_dir}/logs/{timestamp}_mistral_NMF_0.20_final_prune.log")


Cleaning up old processes and freeing GPU memory...
Cleanup complete!
Launching sweeps sequentially per GPU...
Executing on GPU 0: run_gpu0_20260828_123335.sh
Executing on GPU 1: run_gpu1_20260828_123335.sh
Executing on GPU 2: run_gpu2_20260828_123335.sh
Executing on GPU 3: run_gpu3_20260828_123335.sh

ALL PIPELINES LAUNCHED SAFELY WITH NOHUP!
To check progress on GPU 0 (Mistral NMF 20%):
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260828_123335_mistral_NMF_0.20_final_prune.log


In [1]:
import subprocess
import os
import time
from datetime import datetime

# ================= CRITICAL FIX: Auto-Kill Previous Runs =================
print("Cleaning up old processes and freeing GPU memory...")
os.system("pkill -u isea28 -f 'sarathi_main'")
os.system("pkill -u isea28 -f 'sarathi_eval'")
time.sleep(3) # Wait for GPUs to clear
print("Cleanup complete!")

# ================= Configuration =================
work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
c4_path = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)
os.makedirs(f"{work_dir}/experiments", exist_ok=True) # Ensure experiments dir exists

# Organize tasks by GPU. Each list runs SEQUENTIALLY on its GPU to prevent OOM
tasks_by_gpu = {
    # GPU 0: Mistral (needs wikitext & obs-damping 0.01)
    0: [
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.20", "E", "NMF", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.40", "E", "NMF", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.20", "B", "Wanda", "--calib-dataset wikitext --obs-damping 0.01"),
        ("mistralai/Mistral-7B-v0.1", "mistral", "0.40", "B", "Wanda", "--calib-dataset wikitext --obs-damping 0.01")
    ],
    # GPU 1: LLaMA-3 (needs wikitext)
    1: [
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "E", "NMF", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "E", "NMF", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "B", "Wanda", "--calib-dataset wikitext"),
        ("meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "B", "Wanda", "--calib-dataset wikitext")
    ],
    # GPU 2: OPT NMF (needs c4 AND explicitly rank 16)
    2: [
        ("facebook/opt-6.7b", "opt", "0.20", "E", "NMF", "--calib-dataset c4 --nmf-rank 16"),
        ("facebook/opt-6.7b", "opt", "0.40", "E", "NMF", "--calib-dataset c4 --nmf-rank 16")
    ],
    # GPU 3: OPT WANDA (needs wikitext)
    3: [
        ("facebook/opt-6.7b", "opt", "0.20", "B", "Wanda", "--calib-dataset wikitext"),
        ("facebook/opt-6.7b", "opt", "0.40", "B", "Wanda", "--calib-dataset wikitext")
    ]
}

scripts_to_launch = []

for gpu, tasks in tasks_by_gpu.items():
    script_content = f"""#!/bin/bash
export CUDA_VISIBLE_DEVICES={gpu}
export HF_HUB_OFFLINE=0
export SARATHI_C4_PATH={c4_path}
cd {work_dir}

"""
    for model, alias, r, v_code, v_name, extra_args in tasks:
        run_name = f"{alias}_{v_name}_rank16_{r}_final" if alias == "opt" else f"{alias}_{v_name}_{r}_final"
        out_dir = f"pruned_models/{run_name}"
        
        script_content += f"echo 'Starting Pruning: {run_name} on GPU {gpu}'\n"
        script_content += f"{python_exec} sarathi_main.py --multi-gpu --model {model} --structured-ratio {r} --variant {v_code} --adaptive --obs-reconstruct {extra_args} --save-dir {out_dir} > logs/{timestamp}_{run_name}_prune.log 2>&1\n"
        
        script_content += f"echo 'Starting Evaluation: {run_name}'\n"
        script_content += f"{python_exec} sarathi_eval.py --multi-gpu --model {model} --model-path {out_dir} --sobp-tasks --ppl > logs/{timestamp}_{run_name}_eval.log 2>&1\n"
        script_content += f"echo 'Finished: {run_name}'\n\n"

    # Changed path to save in experiments folder
    script_path = f"{work_dir}/experiments/run_gpu{gpu}_{timestamp}.sh"
    with open(script_path, "w") as f: 
        f.write(script_content)
    os.chmod(script_path, 0o755)
    scripts_to_launch.append((gpu, script_path))

# ================= Execution =================
print("Launching sweeps sequentially per GPU...")
for gpu, script in scripts_to_launch:
    print(f"Executing on GPU {gpu}: {os.path.basename(script)}")
    subprocess.Popen(["nohup", "bash", script], cwd=work_dir, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)

print("\nALL PIPELINES LAUNCHED SAFELY WITH NOHUP & EFFICIENT MEMORY MAPPING!")
print("To check progress on GPU 0 (Mistral NMF 20%):")
print(f"!tail -f {work_dir}/logs/{timestamp}_mistral_NMF_0.20_final_prune.log")


Cleaning up old processes and freeing GPU memory...
Cleanup complete!
Launching sweeps sequentially per GPU...
Executing on GPU 0: run_gpu0_20260828_124737.sh
Executing on GPU 1: run_gpu1_20260828_124737.sh
Executing on GPU 2: run_gpu2_20260828_124737.sh
Executing on GPU 3: run_gpu3_20260828_124737.sh

ALL PIPELINES LAUNCHED SAFELY WITH NOHUP & EFFICIENT MEMORY MAPPING!
To check progress on GPU 0 (Mistral NMF 20%):
!tail -f /nlsasfs/home/isea/isea28/SARATHI-E9FD/logs/20260828_124737_mistral_NMF_0.20_final_prune.log


In [9]:
import subprocess
import os
import time
from datetime import datetime

print("Cleaning up old processes and freeing GPU memory...")
os.system("pkill -u isea28 -f 'sarathi_main'")
os.system("pkill -u isea28 -f 'sarathi_eval'")
time.sleep(3) 

work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

def launch_task(gpu_id, hf_model, alias, ratio, variant, name, extra_args=""):
    model_path = f"{work_dir}/pruned_models/{alias}_{name}_rank16_{ratio}_final"
    prune_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_prune.log"
    eval_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_eval.log"
    
    prune_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_main.py --model {hf_model} --variant {variant} --structured-ratio {ratio} --probe-sigma 0.10 --adaptive --min-keep 0.5 --obs-reconstruct --n-calib 128 --calib-seq-len 2048 {extra_args} --save-dir {model_path} > {prune_log} 2>&1"
    
    eval_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_eval.py --multi-gpu --model {hf_model} --model-path {model_path} --sobp-tasks --ppl > {eval_log} 2>&1"
    
    full_cmd = f"({prune_cmd} && {eval_cmd})"
    print(f"Launching {name} {ratio} on GPU {gpu_id}...")
    
    # Yaha cwd=work_dir aur environment variables add kiye gaye hain
    env = os.environ.copy()
    env["HF_HUB_OFFLINE"] = "0"
    env["SARATHI_C4_PATH"] = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"
    
    return subprocess.Popen(full_cmd, shell=True, executable="/bin/bash", cwd=work_dir, env=env)

# --- PHASE 1: NMF RUNS (GPU 0 & GPU 1) ---
print("--- Starting Phase 1: NMF Runs ---")
p1 = launch_task(0, "facebook/opt-6.7b", "opt", "0.20", "E", "NMF", f"--yopo-rank 16 --yopo-iters 100 --calib-dataset c4")
p2 = launch_task(1, "facebook/opt-6.7b", "opt", "0.40", "E", "NMF", f"--yopo-rank 16 --yopo-iters 100 --calib-dataset c4")

print("Waiting for Phase 1 to finish... (This will take time due to original OBS loop)")
p1.wait()
p2.wait()
print("--- Phase 1 Complete! ---")

# --- PHASE 2: WANDA RUNS (GPU 0 & GPU 1) ---
print("--- Starting Phase 2: Wanda Runs ---")
# Wanda now uses c4 dataset to match your original sweep script!
p3 = launch_task(0, "facebook/opt-6.7b", "opt", "0.20", "B", "Wanda", "--calib-dataset c4")
p4 = launch_task(1, "facebook/opt-6.7b", "opt", "0.40", "B", "Wanda", "--calib-dataset c4")

print("Waiting for Phase 2 to finish...")
p3.wait()
p4.wait()
print("All tasks finished successfully!")


Cleaning up old processes and freeing GPU memory...
--- Starting Phase 1: NMF Runs ---
Launching NMF 0.20 on GPU 0...
Launching NMF 0.40 on GPU 1...
Waiting for Phase 1 to finish... (This will take time due to original OBS loop)
--- Phase 1 Complete! ---
--- Starting Phase 2: Wanda Runs ---
Launching Wanda 0.20 on GPU 0...
Launching Wanda 0.40 on GPU 1...
Waiting for Phase 2 to finish...
✅ All tasks finished successfully!


In [ ]:
import subprocess
import os
import time
from datetime import datetime

print("Cleaning up old processes and freeing GPU memory...")
os.system("pkill -u isea28 -f 'sarathi_main'")
os.system("pkill -u isea28 -f 'sarathi_eval'")
print("Waiting 5 seconds for GPUs to clear...")
time.sleep(5)

work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

def launch_task(gpu_id, hf_model, alias, ratio, variant, name, extra_args=""):
    model_path = f"{work_dir}/pruned_models/{alias}_{name}_rank16_{ratio}_final"
    prune_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_prune.log"
    eval_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_eval.log"
    
    # Original EMNLP arguments
    prune_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_main.py --model {hf_model} --variant {variant} --structured-ratio {ratio} --probe-sigma 0.10 --adaptive --min-keep 0.5 --obs-reconstruct --n-calib 128 --calib-seq-len 2048 {extra_args} --save-dir {model_path} > {prune_log} 2>&1"
    
    eval_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_eval.py --multi-gpu --model {hf_model} --model-path {model_path} --sobp-tasks --ppl > {eval_log} 2>&1"
    
    full_cmd = f"({prune_cmd} && {eval_cmd})"
    print(f"Launching {name} {ratio} on GPU {gpu_id} (Logs: {timestamp}_{alias}_{name}_..._prune.log)...")
    
    env = os.environ.copy()
    env["HF_HUB_OFFLINE"] = "0"
    env["SARATHI_C4_PATH"] = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"
    
    # cwd=work_dir added to fix Jupyter path issues
    return subprocess.Popen(full_cmd, shell=True, executable="/bin/bash", cwd=work_dir, env=env)

# --- PHASE 1: NMF RUNS (GPU 0 & GPU 1) ---
print("--- Starting Phase 1: NMF Runs ---")
# CORRECTED: Uses --nmf-rank and --nmf-iters instead of yopo-*
p1 = launch_task(0, "facebook/opt-6.7b", "opt", "0.20", "E", "NMF", f"--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")
p2 = launch_task(1, "facebook/opt-6.7b", "opt", "0.40", "E", "NMF", f"--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")

print("Waiting for Phase 1 to finish... (This will take time due to original OBS loop)")
p1.wait()
p2.wait()
print("--- Phase 1 Complete! ---")

# --- PHASE 2: WANDA RUNS (GPU 0 & GPU 1) ---
print("--- Starting Phase 2: Wanda Runs ---")
# FIX: The original OPT Wanda was calibrated on c4, NOT wikitext!
p3 = launch_task(0, "facebook/opt-6.7b", "opt", "0.20", "B", "Wanda", "--calib-dataset c4")
p4 = launch_task(1, "facebook/opt-6.7b", "opt", "0.40", "B", "Wanda", "--calib-dataset c4")

print("Waiting for Phase 2 to finish...")
p3.wait()
p4.wait()
print("All tasks finished successfully!")


Cleaning up old processes and freeing GPU memory...
Waiting 5 seconds for GPUs to clear...
--- Starting Phase 1: NMF Runs ---
Launching NMF 0.20 on GPU 0 (Logs: 20260828_161706_opt_NMF_..._prune.log)...
Launching NMF 0.40 on GPU 1 (Logs: 20260828_161706_opt_NMF_..._prune.log)...
Waiting for Phase 1 to finish... (This will take time due to original OBS loop)


In [ ]:
import subprocess
import os
import time
from datetime import datetime

work_dir = "/nlsasfs/home/isea/isea28/SARATHI-E9FD"
python_exec = "/nlsasfs/home/isea/isea28/venv_jupyter/bin/python"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(f"{work_dir}/logs", exist_ok=True)
os.makedirs(f"{work_dir}/pruned_models", exist_ok=True)

def launch_task(gpu_id, hf_model, alias, ratio, variant, name, extra_args=""):
    model_path = f"{work_dir}/pruned_models/{alias}_{name}_rank16_{ratio}_final"
    prune_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_prune.log"
    eval_log = f"{work_dir}/logs/{timestamp}_{alias}_{name}_rank16_{ratio}_final_eval.log"
    
    prune_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_main.py --model {hf_model} --variant {variant} --structured-ratio {ratio} --probe-sigma 0.10 --adaptive --min-keep 0.5 --obs-reconstruct --n-calib 128 --calib-seq-len 2048 {extra_args} --save-dir {model_path} > {prune_log} 2>&1"
    
    eval_cmd = f"CUDA_VISIBLE_DEVICES={gpu_id} {python_exec} sarathi_eval.py --multi-gpu --model {hf_model} --model-path {model_path} --sobp-tasks --ppl > {eval_log} 2>&1"
    
    full_cmd = f"({prune_cmd} && {eval_cmd})"
    print(f"Launching {alias} {name} {ratio} on GPU {gpu_id}...")
    
    env = os.environ.copy()
    env["HF_HUB_OFFLINE"] = "0"
    env["SARATHI_C4_PATH"] = "/nlsasfs/home/isea/isea11/Sankar/Dhruv/c4_calibration_subset.json"
    
    return subprocess.Popen(full_cmd, shell=True, executable="/bin/bash", cwd=work_dir, env=env)

print("--- Starting Mistral & LLaMA Runs (Sequential 2 at a time) ---")

# --- BATCH 1 (GPUs 0 & 1) ---
p1 = launch_task(0, "mistralai/Mistral-7B-v0.1", "mistral", "0.20", "E", "NMF", "--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")
p2 = launch_task(1, "mistralai/Mistral-7B-v0.1", "mistral", "0.40", "E", "NMF", "--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")
print("Waiting for Mistral NMF to finish...")
p1.wait()
p2.wait()

# --- BATCH 2 (GPUs 2 & 3) ---
p3 = launch_task(2, "meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "E", "NMF", "--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")
p4 = launch_task(3, "meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "E", "NMF", "--nmf-rank 16 --nmf-iters 100 --calib-dataset c4")
print("Waiting for LLaMA-3 NMF to finish...")
p3.wait()
p4.wait()

# --- BATCH 3 (GPUs 0 & 1) ---
p5 = launch_task(0, "mistralai/Mistral-7B-v0.1", "mistral", "0.20", "B", "Wanda", "--calib-dataset wikitext")
p6 = launch_task(1, "mistralai/Mistral-7B-v0.1", "mistral", "0.40", "B", "Wanda", "--calib-dataset wikitext")
print("Waiting for Mistral Wanda to finish...")
p5.wait()
p6.wait()

# --- BATCH 4 (GPUs 2 & 3) ---
p7 = launch_task(2, "meta-llama/Meta-Llama-3-8B", "llama3", "0.20", "B", "Wanda", "--calib-dataset wikitext")
p8 = launch_task(3, "meta-llama/Meta-Llama-3-8B", "llama3", "0.40", "B", "Wanda", "--calib-dataset wikitext")
print("Waiting for LLaMA-3 Wanda to finish...")
p7.wait()
p8.wait()

print("All Mistral and LLaMA tasks finished successfully!")


--- Starting Mistral & LLaMA Runs (Sequential 2 at a time) ---
Launching mistral NMF 0.20 on GPU 0...
Launching mistral NMF 0.40 on GPU 1...
Waiting for Mistral NMF to finish...


In [77]:
!nvidia-smi

Sat Aug 29 18:50:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   54C    P0            331W /  400W |   23431MiB /  40960MiB |     98%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----